# MLOps Platform — Pipeline Runbook

This notebook provides operational commands for the MDAA MLOps pipeline.

The pipeline is defined via **static YAML configs** in `mdaa-config/pipeline.yaml` and deployed by
the training CodeBuild project via `mdaa deploy`. This notebook does NOT define or create the pipeline;
it provides commands for inspecting and operating the deployed pipeline.

## Architecture

1. **Training CodePipeline** (`mdaa deploy`) creates a SageMaker Pipeline with steps:
   - `PreprocessAbaloneData` — feature engineering (XGBoost container)
   - `TrainAbaloneModel` — XGBoost regression training
   - `RegisterAbaloneModel` — registers model in Model Package Group
2. **Deploy CodePipeline** (triggered by model approval via EventBridge) deploys an endpoint + monitoring.
3. **Batch Inference CodePipeline** (optional) runs batch transform on approved models.

## Setup

Set the variables below to match your deployment.

In [ ]:
import boto3
import sagemaker

# TODO: Set these to match your mdaa.yaml organization and project name
MDAA_ORG = "<your-org-name>"
PROJECT_NAME = "<your-project-name>"
REGION = boto3.Session().region_name

sm_client = boto3.client("sagemaker", region_name=REGION)
ssm_client = boto3.client("ssm", region_name=REGION)
print(f"Region: {REGION}, Org: {MDAA_ORG}, Project: {PROJECT_NAME}")

## Lookup Pipeline Resources from SSM

In [ ]:
def ssm_get(path):
    try:
        return ssm_client.get_parameter(Name=path)["Parameter"]["Value"]
    except ssm_client.exceptions.ParameterNotFound:
        return f"NOT FOUND: {path}"

pipeline_name = ssm_get(f"/{MDAA_ORG}/mlops/pipeline/sm-pipeline/{PROJECT_NAME}/pipeline-name")
model_bucket = ssm_get(f"/{MDAA_ORG}/mlops/pipeline/bucket/pipeline-{PROJECT_NAME}/name")
mpg_name = ssm_get(f"/{MDAA_ORG}/mlops/core/model-training/{PROJECT_NAME}/model-package-group-name")

print(f"Pipeline Name: {pipeline_name}")
print(f"Model Bucket:  {model_bucket}")
print(f"Model Package Group: {mpg_name}")

## List Pipeline Executions

In [ ]:
executions = sm_client.list_pipeline_executions(PipelineName=pipeline_name, MaxResults=5)
for ex in executions["PipelineExecutionSummaries"]:
    print(f"{ex['PipelineExecutionArn'].split('/')[-1]}  status={ex['PipelineExecutionStatus']}  {ex.get('StartTime', '')}")

## List Model Packages (Approved)

In [ ]:
packages = sm_client.list_model_packages(
    ModelPackageGroupName=mpg_name,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)
for pkg in packages["ModelPackageSummaryList"]:
    print(f"{pkg['ModelPackageArn'].split('/')[-1]}  status={pkg['ModelApprovalStatus']}  {pkg.get('CreationTime', '')}")

## Start a New Pipeline Execution

**WARNING**: This will start a training job that incurs costs. Only run when intended.

In [ ]:
# Uncomment to start a new execution:
# response = sm_client.start_pipeline_execution(
#     PipelineName=pipeline_name,
#     PipelineExecutionDisplayName="notebook-run",
# )
# print(f"Started: {response['PipelineExecutionArn']}")